<a href="https://colab.research.google.com/github/muhmmadabuzar-svg/Dark-Pattern-Thesis/blob/main/05_rule_base_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openpyxl

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
input_file = "/content/drive/MyDrive/Dark Pattern Thesis/ocr_outputs/visual_signal_coding_thesis_completed.xlsx"

output_file = "/content/drive/MyDrive/Dark Pattern Thesis/ocr_outputs/first_batch_rule_based_results_strict.xlsx"

summary_file = "/content/drive/MyDrive/Dark Pattern Thesis/ocr_outputs/first_batch_rule_based_summary_strict.xlsx"

In [ ]:
URGENCY_KEYWORDS = [
    # Scarcity
    'only', 'left', 'last one', 'last chance', 'almost gone',
    'almost sold out', 'low stock', 'selling fast', 'high demand',
    'limited stock', 'limited availability', 'limited quantity',
    # Time pressure
    'ends', 'ends in', 'deal ends', 'sale ends', 'offer ends',
    'expires', 'expiring', 'hurry', 'today only', 'flash sale',
    'limited time', 'time-limited', 'order within', 'order in',
    'get it by', 'lock in', 'holding your booking',
    # Social proof urgency
    'people viewing', 'others looking', 'trending',
    'bought in past', 'sold', 'early bird', 'claimed',
]

MISDIRECTION_KEYWORDS = [
    # Pricing manipulation
    'was', 'list price', 'typical price', 'rrp', 'msrp',
    'save', 'you save', 'deal', 'special deal', 'spotlight deal',
    'superdeals', '% off', 'off', 'discount', 'coupon',
    'now only', 'now just', 'best price', 'price match',
    # Anchoring
    'compare at', 'original price', 'before',
    # Visual manipulation signals in text
    'recommended', 'popular', 'best seller', 'top pick',
    'most popular', 'bestselling', 'editor', "customer's choice",
]

OBSTRUCTION_KEYWORDS = [
    # Forced actions
    'register', 'sign up', 'create account', 'sign in to',
    'login to', 'log in to', 'join', 'subscribe',
    # Confirmshaming
    'no thanks', 'no, i', "i don't want", 'decline',
    'i prefer full price', 'i hate saving', 'i hate discounts',
    # Forced app
    'get the app', 'download the app', 'open in app',
    'use our app', 'mobile app', 'app only',
    # Hidden terms
    'by continuing', 'by clicking', 'i agree',
    'terms and conditions', 'shipping elsewhere',
]

# ── VISUAL SIGNAL RULES ───────────────────────────────────────────────────────
# These map directly to your manually coded columns
# column name → which pattern it signals

VISUAL_URGENCY_SIGNALS = [
    'countdown_timer_visible',
    'low_stock_text_visible',
    'limited_time_badge_visible',
    'urgency_text_visually_emphasized',
    'large_sale_percentage',
]

VISUAL_MISDIRECTION_SIGNALS = [
    'large_discount_banner',
    'crossed_out_price',
    'price_anchor_visible',
    'dominant_buy_button',
    'sale_color_emphasis',
]

VISUAL_OBSTRUCTION_SIGNALS = [
    'popup_present',
    'popup_blocks_content',
    'dimmed_background',
    'small_close_button',
    'forced_signup_or_register',
    'app_download_prompt',
]

print('✅ Detection rules defined')
print(f'   Urgency keywords:      {len(URGENCY_KEYWORDS)}')
print(f'   Misdirection keywords: {len(MISDIRECTION_KEYWORDS)}')
print(f'   Obstruction keywords:  {len(OBSTRUCTION_KEYWORDS)}')
print(f'   Visual urgency signals:      {len(VISUAL_URGENCY_SIGNALS)}')
print(f'   Visual misdirection signals: {len(VISUAL_MISDIRECTION_SIGNALS)}')
print(f'   Visual obstruction signals:  {len(VISUAL_OBSTRUCTION_SIGNALS)}')

✅ Detection rules defined
   Urgency keywords:      36
   Misdirection keywords: 30
   Obstruction keywords:  26
   Visual urgency signals:      5
   Visual misdirection signals: 5
   Visual obstruction signals:  6


In [ ]:
df = pd.read_excel(input_file, sheet_name='Dark Pattern Coding', header=1)
df.columns = df.columns.str.replace(r'[()\./]', '', regex=True).str.strip().str.lower().str.replace(' ', '_')

print(f'✅ Loaded DataFrame with {len(df)} rows and {len(df.columns)} columns.')

results = []

for idx, row in df.iterrows():
    row_dict = row.to_dict()

    # Run all three detectors
    urgency     = detect_pattern(row_dict, URGENCY_KEYWORDS,
                                  VISUAL_URGENCY_SIGNALS,
                                  'fake_urgency',
                                  existing_col='fake_urgency_detected')

    misdirection = detect_pattern(row_dict, MISDIRECTION_KEYWORDS,
                                   VISUAL_MISDIRECTION_SIGNALS,
                                   'visual_misdirection',
                                   existing_col='visual_misdirection_detected')

    obstruction  = detect_pattern(row_dict, OBSTRUCTION_KEYWORDS,
                                   VISUAL_OBSTRUCTION_SIGNALS,
                                   'obstruction',
                                   existing_col='obstruction_detected')

    # Determine overall dominant pattern
    scores = {
        'fake_urgency':       urgency['confidence'],
        'visual_misdirection': misdirection['confidence'],
        'obstruction':         obstruction['confidence'],
    }
    dominant = max(scores, key=scores.get) if max(scores.values()) > 0 else 'none'

    # Count total signals fired
    total_signals = (
        len(urgency['keyword_evidence'].split(',') if urgency['keyword_evidence'] else []) +
        len(urgency['visual_evidence'].split(',') if urgency['visual_evidence'] else []) +
        len(misdirection['keyword_evidence'].split(',') if misdirection['keyword_evidence'] else []) +
        len(misdirection['visual_evidence'].split(',') if misdirection['visual_evidence'] else []) +
        len(obstruction['keyword_evidence'].split(',') if obstruction['keyword_evidence'] else []) +
        len(obstruction['visual_evidence'].split(',') if obstruction['visual_evidence'] else [])
    )

    results.append({
        # ── Identity ──────────────────────────────────────────────────────
        'image_id':            row_dict.get('image_id', idx+1),
        'filename':            row_dict.get('filename', ''),
        'website':             row_dict.get('website', ''),
        'expected_category':   row_dict.get('expected_category', ''),
        'ocr_quality':         row_dict.get('ocr_quality', ''),

        # ── OCR text (cleaned short version) ──────────────────────────────
        'ocr_text_preview':    str(row_dict.get('ocr_text_clean', ''))[:150],

        # ── FAKE URGENCY RESULTS ──────────────────────────────────────────
        'RB_urgency_detected':     'Yes' if urgency['detected'] else 'No',
        'RB_urgency_confidence':   urgency['confidence'],
        'RB_urgency_kw_evidence':  urgency['keyword_evidence'],
        'RB_urgency_vis_evidence': urgency['visual_evidence'],
        'RB_urgency_explanation':  urgency['explanation'],
        'urgency_manual_vs_rb':    urgency['agrees_with_manual'],

        # ── VISUAL MISDIRECTION RESULTS ───────────────────────────────────
        'RB_misdirect_detected':     'Yes' if misdirection['detected'] else 'No',
        'RB_misdirect_confidence':   misdirection['confidence'],
        'RB_misdirect_kw_evidence':  misdirection['keyword_evidence'],
        'RB_misdirect_vis_evidence': misdirection['visual_evidence'],
        'RB_misdirect_explanation':  misdirection['explanation'],
        'misdirect_manual_vs_rb':    misdirection['agrees_with_manual'],

        # ── OBSTRUCTION RESULTS ───────────────────────────────────────────
        'RB_obstruct_detected':     'Yes' if obstruction['detected'] else 'No',
        'RB_obstruct_confidence':   obstruction['confidence'],
        'RB_obstruct_kw_evidence':  obstruction['keyword_evidence'],
        'RB_obstruct_vis_evidence': obstruction['visual_evidence'],
        'RB_obstruct_explanation':  obstruction['explanation'],
        'obstruct_manual_vs_rb':    obstruction['agrees_with_manual'],

        # ── OVERALL SUMMARY ───────────────────────────────────────────────
        'dominant_pattern':    dominant,
        'total_signals_fired': total_signals,
        'multi_pattern':       'Yes' if sum([
            urgency['detected'],
            misdirection['detected'],
            obstruction['detected']
        ]) > 1 else 'No',
    })

results_df = pd.DataFrame(results)

print(f'✅ Detection complete — {len(results_df)} rows processed')
print(f'')
print(f'📊 DETECTION RESULTS SUMMARY')
print(f'   Fake Urgency detected:      {(results_df["RB_urgency_detected"]=="Yes").sum()} / {len(results_df)}')
print(f'   Visual Misdirection detected:{(results_df["RB_misdirect_detected"]=="Yes").sum()} / {len(results_df)}')
print(f'   Obstruction detected:        {(results_df["RB_obstruct_detected"]=="Yes").sum()} / {len(results_df)}')
print(f'   Multi-pattern images:        {(results_df["multi_pattern"]=="Yes").sum()} / {len(results_df)}')
print(f'')
print(f'🤝 AGREEMENT WITH YOUR MANUAL LABELS')
for col, label in [
    ('urgency_manual_vs_rb',  'Urgency'),
    ('misdirect_manual_vs_rb','Misdirection'),
    ('obstruct_manual_vs_rb', 'Obstruction'),
]:
    matches = (results_df[col] == 'Match').sum()
    total = results_df[col].notna().sum()
    pct = matches/total*100 if total > 0 else 0
    print(f'   {label}: {matches}/{total} agree ({pct:.1f}%)')

✅ Loaded DataFrame with 50 rows and 26 columns.
✅ Detection complete — 50 rows processed

📊 DETECTION RESULTS SUMMARY
   Fake Urgency detected:      0 / 50
   Visual Misdirection detected:0 / 50
   Obstruction detected:        0 / 50
   Multi-pattern images:        0 / 50

🤝 AGREEMENT WITH YOUR MANUAL LABELS
   Urgency: 0/0 agree (0.0%)
   Misdirection: 0/0 agree (0.0%)
   Obstruction: 0/0 agree (0.0%)


In [ ]:
def compute_metrics(rb_col, manual_col, df_in):
    """Compute TP, FP, FN, TN, Precision, Recall, F1"""
    rb  = df_in[rb_col].str.lower().isin(['yes'])
    man = df_in[manual_col].str.strip().str.lower().isin(['yes'])

    tp = (rb & man).sum()
    fp = (rb & ~man).sum()
    fn = (~rb & man).sum()
    tn = (~rb & ~man).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    accuracy  = (tp + tn) / len(df_in) if len(df_in) > 0 else 0.0

    return {
        'TP': int(tp), 'FP': int(fp),
        'FN': int(fn), 'TN': int(tn),
        'Precision': round(precision, 3),
        'Recall':    round(recall, 3),
        'F1':        round(f1, 3),
        'Accuracy':  round(accuracy, 3),
    }

# Only compute for rows where manual label exists in original df
eval_df = results_df.copy()

# Extract manual labels from the 'Pattern Category' column in the original df
# and convert them to 'Yes'/'No' strings for consistency with compute_metrics
eval_df['manual_urgency'] = df['unnamed:_3'].astype(str).str.contains('fake_urgency', na=False).map({True: 'Yes', False: 'No'})
eval_df['manual_misdirect'] = df['unnamed:_3'].astype(str).str.contains('visual_misdirection', na=False).map({True: 'Yes', False: 'No'})
eval_df['manual_obstruction'] = df['unnamed:_3'].astype(str).str.contains('obstruction', na=False).map({True: 'Yes', False: 'No'})

metrics = {}
for name, rb_col, man_col in [
    ('Fake Urgency',       'RB_urgency_detected',  'manual_urgency'),
    ('Vis. Misdirection',  'RB_misdirect_detected','manual_misdirect'),
    ('Obstruction',        'RB_obstruct_detected', 'manual_obstruction'),
]:
    metrics[name] = compute_metrics(rb_col, man_col, eval_df)

print('📐 EVALUATION METRICS — Rule-Based Detection vs Manual Labels')
print('   (This goes directly into your thesis results chapter)')
print()
print(f'{"Pattern":<22} {"TP":>4} {"FP":>4} {"FN":>4} {"TN":>4}   {"Prec":>6} {"Rec":>6} {"F1":>6} {"Acc":>6}')
print('─' * 72)
for name, m in metrics.items():
    print(f'{name:<22} {m["TP"]:>4} {m["FP"]:>4} {m["FN"]:>4} {m["TN"]:>4}   '
          f'{m["Precision"]:>6.3f} {m["Recall"]:>6.3f} {m["F1"]:>6.3f} {m["Accuracy"]:>6.3f}')
print()
print('   TP=True Positive, FP=False Positive, FN=False Negative, TN=True Negative')
print('   Prec=Precision, Rec=Recall, F1=F1-Score, Acc=Accuracy')

📐 EVALUATION METRICS — Rule-Based Detection vs Manual Labels
   (This goes directly into your thesis results chapter)

Pattern                  TP   FP   FN   TN     Prec    Rec     F1    Acc
────────────────────────────────────────────────────────────────────────
Fake Urgency              0    0   37   13    0.000  0.000  0.000  0.260
Vis. Misdirection         0    0    3   47    0.000  0.000  0.000  0.940
Obstruction               0    0    7   43    0.000  0.000  0.000  0.860

   TP=True Positive, FP=False Positive, FN=False Negative, TN=True Negative
   Prec=Precision, Rec=Recall, F1=F1-Score, Acc=Accuracy


In [ ]:
import re

def get_text(row):
    """Get OCR text from row, prefer clean version"""
    text = str(row.get('ocr_text_clean', '') or '')
    if len(text) < 10:  # fallback to raw if clean is too short
        text = str(row.get('ocr_text_raw', '') or '')
    return text.lower()


def find_keyword_matches(text, keyword_list):
    """Find which keywords from the list appear in the text.
    Returns list of matched keywords."""
    matched = []
    for kw in keyword_list:
        # Use word boundary matching for short keywords to avoid false positives
        if len(kw) <= 4:
            pattern = r'\b' + re.escape(kw) + r'\b'
        else:
            pattern = re.escape(kw)
        if re.search(pattern, text):
            matched.append(kw)
    return matched


def get_visual_signals(row, signal_columns):
    """Get list of visual signal columns that are 'Yes' in this row."""
    fired = []
    for col in signal_columns:
        val = str(row.get(col, 'No') or 'No').strip().lower()
        if val in ('yes', '1', 'true', 'y'):
            fired.append(col)
    return fired


def compute_confidence(keyword_matches, visual_matches,
                        n_keywords, n_visual_cols):
    """Compute confidence score 0.0–1.0 based on how many signals fired.

    Formula:
    - Keyword contribution: up to 0.5 (capped at 3 keywords = max)
    - Visual contribution:  up to 0.5 (capped at 2 signals = max)
    """
    kw_score = min(len(keyword_matches) / 3, 1.0) * 0.5
    vis_score = min(len(visual_matches) / 2, 1.0) * 0.5
    return round(kw_score + vis_score, 2)


def detect_pattern(row, keywords, visual_cols,
                   label_name, existing_col=None):
    """Run rule-based detection for one pattern category.

    Returns dict with:
    - detected: True/False
    - confidence: 0.0–1.0
    - keyword_evidence: list of matched keywords
    - visual_evidence: list of fired visual signals
    - rule_explanation: human-readable explanation
    - agrees_with_manual: whether result matches your manual label
    """
    text = get_text(row)
    kw_matches = find_keyword_matches(text, keywords)
    vis_matches = get_visual_signals(row, visual_cols)

    detected = len(kw_matches) > 0 or len(vis_matches) > 0
    confidence = compute_confidence(kw_matches, vis_matches,
                                     len(keywords), len(visual_cols))

    # Build human-readable explanation
    parts = []
    if kw_matches:
        parts.append(f"Text signals: {', '.join(kw_matches[:4])}")
    if vis_matches:
        clean_vis = [v.replace('_', ' ') for v in vis_matches]
        parts.append(f"Visual signals: {', '.join(clean_vis[:3])}")
    if not parts:
        parts.append('No signals detected')
    explanation = ' | '.join(parts)

    # Compare with manual label if column exists
    agrees = None
    if existing_col and existing_col in row:
        manual = str(row[existing_col] or 'No').strip().lower()
        manual_bool = manual in ('yes', '1', 'true', 'y')
        agrees = 'Match' if (detected == manual_bool) else 'Mismatch'

    return {
        'detected': detected,
        'confidence': confidence,
        'keyword_evidence': ', '.join(kw_matches) if kw_matches else '',
        'visual_evidence': ', '.join([v.replace('_',' ') for v in vis_matches]),
        'explanation': explanation,
        'agrees_with_manual': agrees,
    }


print('✅ Detection functions defined')

✅ Detection functions defined


In [ ]:
df = pd.read_excel(input_file, sheet_name='Dark Pattern Coding', header=1)
df.columns = df.columns.str.replace(r'[()\./]', '', regex=True).str.strip().str.lower().str.replace(' ', '_')

print(f'✅ Loaded DataFrame with {len(df)} rows and {len(df.columns)} columns.')

results = []

for idx, row in df.iterrows():
    row_dict = row.to_dict()

    # Run all three detectors
    urgency     = detect_pattern(row_dict, URGENCY_KEYWORDS,
                                  VISUAL_URGENCY_SIGNALS,
                                  'fake_urgency',
                                  existing_col='fake_urgency_detected')

    misdirection = detect_pattern(row_dict, MISDIRECTION_KEYWORDS,
                                   VISUAL_MISDIRECTION_SIGNALS,
                                   'visual_misdirection',
                                   existing_col='visual_misdirection_detected')

    obstruction  = detect_pattern(row_dict, OBSTRUCTION_KEYWORDS,
                                   VISUAL_OBSTRUCTION_SIGNALS,
                                   'obstruction',
                                   existing_col='obstruction_detected')

    # Determine overall dominant pattern
    scores = {
        'fake_urgency':       urgency['confidence'],
        'visual_misdirection': misdirection['confidence'],
        'obstruction':         obstruction['confidence'],
    }
    dominant = max(scores, key=scores.get) if max(scores.values()) > 0 else 'none'

    # Count total signals fired
    total_signals = (
        len(urgency['keyword_evidence'].split(',') if urgency['keyword_evidence'] else []) +
        len(urgency['visual_evidence'].split(',') if urgency['visual_evidence'] else []) +
        len(misdirection['keyword_evidence'].split(',') if misdirection['keyword_evidence'] else []) +
        len(misdirection['visual_evidence'].split(',') if misdirection['visual_evidence'] else []) +
        len(obstruction['keyword_evidence'].split(',') if obstruction['keyword_evidence'] else []) +
        len(obstruction['visual_evidence'].split(',') if obstruction['visual_evidence'] else [])
    )

    results.append({
        # ── Identity ──────────────────────────────────────────────────────
        'image_id':            row_dict.get('image_id', idx+1),
        'filename':            row_dict.get('filename', ''),
        'website':             row_dict.get('website', ''),
        'expected_category':   row_dict.get('expected_category', ''),
        'ocr_quality':         row_dict.get('ocr_quality', ''),

        # ── OCR text (cleaned short version) ──────────────────────────────
        'ocr_text_preview':    str(row_dict.get('ocr_text_clean', ''))[:150],

        # ── FAKE URGENCY RESULTS ──────────────────────────────────────────
        'RB_urgency_detected':     'Yes' if urgency['detected'] else 'No',
        'RB_urgency_confidence':   urgency['confidence'],
        'RB_urgency_kw_evidence':  urgency['keyword_evidence'],
        'RB_urgency_vis_evidence': urgency['visual_evidence'],
        'RB_urgency_explanation':  urgency['explanation'],
        'urgency_manual_vs_rb':    urgency['agrees_with_manual'],

        # ── VISUAL MISDIRECTION RESULTS ───────────────────────────────────
        'RB_misdirect_detected':     'Yes' if misdirection['detected'] else 'No',
        'RB_misdirect_confidence':   misdirection['confidence'],
        'RB_misdirect_kw_evidence':  misdirection['keyword_evidence'],
        'RB_misdirect_vis_evidence': misdirection['visual_evidence'],
        'RB_misdirect_explanation':  misdirection['explanation'],
        'misdirect_manual_vs_rb':    misdirection['agrees_with_manual'],

        # ── OBSTRUCTION RESULTS ───────────────────────────────────────────
        'RB_obstruct_detected':     'Yes' if obstruction['detected'] else 'No',
        'RB_obstruct_confidence':   obstruction['confidence'],
        'RB_obstruct_kw_evidence':  obstruction['keyword_evidence'],
        'RB_obstruct_vis_evidence': obstruction['visual_evidence'],
        'RB_obstruct_explanation':  obstruction['explanation'],
        'obstruct_manual_vs_rb':    obstruction['agrees_with_manual'],

        # ── OVERALL SUMMARY ───────────────────────────────────────────────
        'dominant_pattern':    dominant,
        'total_signals_fired': total_signals,
        'multi_pattern':       'Yes' if sum([
            urgency['detected'],
            misdirection['detected'],
            obstruction['detected']
        ]) > 1 else 'No',
    })

results_df = pd.DataFrame(results)

print(f'✅ Detection complete — {len(results_df)} rows processed')
print(f'')
print(f'📊 DETECTION RESULTS SUMMARY')
print(f'   Fake Urgency detected:      {(results_df["RB_urgency_detected"]=="Yes").sum()} / {len(results_df)}')
print(f'   Visual Misdirection detected:{(results_df["RB_misdirect_detected"]=="Yes").sum()} / {len(results_df)}')
print(f'   Obstruction detected:        {(results_df["RB_obstruct_detected"]=="Yes").sum()} / {len(results_df)}')
print(f'   Multi-pattern images:        {(results_df["multi_pattern"]=="Yes").sum()} / {len(results_df)}')
print(f'')
print(f'🤝 AGREEMENT WITH YOUR MANUAL LABELS')
for col, label in [
    ('urgency_manual_vs_rb',  'Urgency'),
    ('misdirect_manual_vs_rb','Misdirection'),
    ('obstruct_manual_vs_rb', 'Obstruction'),
]:
    matches = (results_df[col] == 'Match').sum()
    total = results_df[col].notna().sum()
    pct = matches/total*100 if total > 0 else 0
    print(f'   {label}: {matches}/{total} agree ({pct:.1f}%)')

✅ Loaded DataFrame with 50 rows and 26 columns.
✅ Detection complete — 50 rows processed

📊 DETECTION RESULTS SUMMARY
   Fake Urgency detected:      0 / 50
   Visual Misdirection detected:0 / 50
   Obstruction detected:        0 / 50
   Multi-pattern images:        0 / 50

🤝 AGREEMENT WITH YOUR MANUAL LABELS
   Urgency: 0/0 agree (0.0%)
   Misdirection: 0/0 agree (0.0%)
   Obstruction: 0/0 agree (0.0%)


In [ ]:
preview_cols = [
    'image_id', 'website', 'expected_category',
    'RB_urgency_detected', 'RB_urgency_confidence',
    'RB_misdirect_detected', 'RB_misdirect_confidence',
    'RB_obstruct_detected', 'RB_obstruct_confidence',
    'dominant_pattern', 'total_signals_fired',
    'urgency_manual_vs_rb', 'misdirect_manual_vs_rb', 'obstruct_manual_vs_rb'
]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 30)

print('🔍 Preview of rule-based detection results (first 10 rows):')
print(results_df[preview_cols].head(10).to_string(index=False))

🔍 Preview of rule-based detection results (first 10 rows):
 image_id website expected_category RB_urgency_detected  RB_urgency_confidence RB_misdirect_detected  RB_misdirect_confidence RB_obstruct_detected  RB_obstruct_confidence dominant_pattern  total_signals_fired urgency_manual_vs_rb misdirect_manual_vs_rb obstruct_manual_vs_rb
        1                                            No                    0.0                    No                      0.0                   No                     0.0             none                    0                 None                   None                  None
        2                                            No                    0.0                    No                      0.0                   No                     0.0             none                    0                 None                   None                  None
        3                                            No                    0.0                    No                

In [ ]:
print('⚠️  MISMATCHES — where rule-based detection disagrees with your manual labels:')
print('   These are worth examining — they reveal rule gaps or annotation edge cases')
print()

for col, label in [
    ('urgency_manual_vs_rb',  'FAKE URGENCY'),
    ('misdirect_manual_vs_rb','VISUAL MISDIRECTION'),
    ('obstruct_manual_vs_rb', 'OBSTRUCTION'),
]:
    mismatches = results_df[results_df[col] == 'Mismatch']
    if len(mismatches) > 0:
        print(f'── {label}: {len(mismatches)} mismatches ──')
        for _, row in mismatches.iterrows():
            rb_col = col.replace('manual_vs_rb', 'detected').replace('urgency_','RB_urgency_').replace('misdirect_','RB_misdirect_').replace('obstruct_','RB_obstruct_')
            # Map to correct column
            detect_map = {
                'urgency_manual_vs_rb': 'RB_urgency_detected',
                'misdirect_manual_vs_rb': 'RB_misdirect_detected',
                'obstruct_manual_vs_rb': 'RB_obstruct_detected',
            }
            exp_map = {
                'urgency_manual_vs_rb': 'RB_urgency_explanation',
                'misdirect_manual_vs_rb': 'RB_misdirect_explanation',
                'obstruct_manual_vs_rb': 'RB_obstruct_explanation',
            }
            print(f'   ID {int(row["image_id"]):02d} | {row["website"]:12s} | '
                  f'Expected: {row["expected_category"]:35s} | '
                  f'RB says: {row[detect_map[col]]:3s} | '
                  f'{row[exp_map[col]][:60]}')
        print()
    else:
        print(f'── {label}: ✅ No mismatches')
        print()

⚠️  MISMATCHES — where rule-based detection disagrees with your manual labels:
   These are worth examining — they reveal rule gaps or annotation edge cases

── FAKE URGENCY: ✅ No mismatches

── VISUAL MISDIRECTION: ✅ No mismatches

── OBSTRUCTION: ✅ No mismatches



In [ ]:
def compute_metrics(rb_col, manual_col, df_in):
    """Compute TP, FP, FN, TN, Precision, Recall, F1"""
    rb  = df_in[rb_col].str.lower().isin(['yes'])
    man = df_in[manual_col].str.strip().str.lower().isin(['yes'])

    tp = (rb & man).sum()
    fp = (rb & ~man).sum()
    fn = (~rb & man).sum()
    tn = (~rb & ~man).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    accuracy  = (tp + tn) / len(df_in) if len(df_in) > 0 else 0.0

    return {
        'TP': int(tp), 'FP': int(fp),
        'FN': int(fn), 'TN': int(tn),
        'Precision': round(precision, 3),
        'Recall':    round(recall, 3),
        'F1':        round(f1, 3),
        'Accuracy':  round(accuracy, 3),
    }

# Only compute for rows where manual label exists in original df
eval_df = results_df.copy()

# Extract manual labels from the 'Pattern Category' column in the original df
# and convert them to 'Yes'/'No' strings for consistency with compute_metrics
eval_df['manual_urgency'] = df['unnamed:_3'].astype(str).str.contains('fake_urgency', na=False).map({True: 'Yes', False: 'No'})
eval_df['manual_misdirect'] = df['unnamed:_3'].astype(str).str.contains('visual_misdirection', na=False).map({True: 'Yes', False: 'No'})
eval_df['manual_obstruction'] = df['unnamed:_3'].astype(str).str.contains('obstruction', na=False).map({True: 'Yes', False: 'No'})

metrics = {}
for name, rb_col, man_col in [
    ('Fake Urgency',       'RB_urgency_detected',  'manual_urgency'),
    ('Vis. Misdirection',  'RB_misdirect_detected','manual_misdirect'),
    ('Obstruction',        'RB_obstruct_detected', 'manual_obstruction'),
]:
    metrics[name] = compute_metrics(rb_col, man_col, eval_df)

print('📐 EVALUATION METRICS — Rule-Based Detection vs Manual Labels')
print('   (This goes directly into your thesis results chapter)')
print()
print(f'{"Pattern":<22} {"TP":>4} {"FP":>4} {"FN":>4} {"TN":>4}   {"Prec":>6} {"Rec":>6} {"F1":>6} {"Acc":>6}')
print('─' * 72)
for name, m in metrics.items():
    print(f'{name:<22} {m["TP"]:>4} {m["FP"]:>4} {m["FN"]:>4} {m["TN"]:>4}   '
          f'{m["Precision"]:>6.3f} {m["Recall"]:>6.3f} {m["F1"]:>6.3f} {m["Accuracy"]:>6.3f}')
print()
print('   TP=True Positive, FP=False Positive, FN=False Negative, TN=True Negative')
print('   Prec=Precision, Rec=Recall, F1=F1-Score, Acc=Accuracy')

📐 EVALUATION METRICS — Rule-Based Detection vs Manual Labels
   (This goes directly into your thesis results chapter)

Pattern                  TP   FP   FN   TN     Prec    Rec     F1    Acc
────────────────────────────────────────────────────────────────────────
Fake Urgency              0    0   37   13    0.000  0.000  0.000  0.260
Vis. Misdirection         0    0    3   47    0.000  0.000  0.000  0.940
Obstruction               0    0    7   43    0.000  0.000  0.000  0.860

   TP=True Positive, FP=False Positive, FN=False Negative, TN=True Negative
   Prec=Precision, Rec=Recall, F1=F1-Score, Acc=Accuracy


In [ ]:
print('🌐 DETECTION BREAKDOWN BY WEBSITE')
print()

for site in results_df['website'].unique():
    site_df = results_df[results_df['website'] == site]
    n = len(site_df)
    u = (site_df['RB_urgency_detected'] == 'Yes').sum()
    m = (site_df['RB_misdirect_detected'] == 'Yes').sum()
    o = (site_df['RB_obstruct_detected'] == 'Yes').sum()
    multi = (site_df['multi_pattern'] == 'Yes').sum()
    print(f'  {site:<14} ({n:2d} images) | '
          f'Urgency:{u:2d} | Misdirection:{m:2d} | Obstruction:{o:2d} | Multi:{multi:2d}')

🌐 DETECTION BREAKDOWN BY WEBSITE

                 (50 images) | Urgency: 0 | Misdirection: 0 | Obstruction: 0 | Multi: 0


In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

output_path = 'rule_based_detection_results.xlsx'

# ── Build summary metrics dataframe ──────────────────────────────────────────
metrics_rows = []
for name, m in metrics.items():
    metrics_rows.append({'Pattern': name, **m})
metrics_df = pd.DataFrame(metrics_rows)

# ── Write both sheets ─────────────────────────────────────────────────────────
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='RB Detection Results', index=False, startrow=2)
    metrics_df.to_excel(writer, sheet_name='Evaluation Metrics', index=False, startrow=2)

# ── Apply formatting ──────────────────────────────────────────────────────────
wb = load_workbook(output_path)

thin = Side(style='thin', color='BDC3C7')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def fill(c): return PatternFill('solid', fgColor=c)
def hdr(cell, bg='1C2833', fg='FFFFFF', sz=10):
    cell.font = Font(name='Arial', bold=True, color=fg, size=sz)
    cell.fill = fill(bg)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = border

def dat(cell, bg=None, fg='2C3E50', bold=False, center=False):
    cell.font = Font(name='Arial', size=9, color=fg, bold=bold)
    if bg: cell.fill = fill(bg)
    cell.alignment = Alignment(horizontal='center' if center else 'left',
                                vertical='top', wrap_text=True)
    cell.border = border

# ── Sheet 1: RB Detection Results ────────────────────────────────────────────
ws = wb['RB Detection Results']
ws.merge_cells('A1:AC1')
t = ws['A1']
t.value = 'Rule-Based Dark Pattern Detection Results — AI-Assisted Dark Pattern Detection Thesis'
t.font = Font(name='Arial', bold=True, size=13, color='FFFFFF')
t.fill = fill('1C2833')
t.alignment = Alignment(horizontal='center', vertical='center')
ws.row_dimensions[1].height = 26

ws.merge_cells('A2:AC2')
s = ws['A2']
s.value = 'DSRM Phase 2 | Textual + Visual Signal Rule-Based Baseline | University of Vaasa 2026'
s.font = Font(name='Arial', italic=True, size=9, color='7F8C8D')
s.fill = fill('F2F3F4')
s.alignment = Alignment(horizontal='center', vertical='center')
ws.row_dimensions[2].height = 14

# Header colours by column group
col_colors = {
    # Identity
    1:'1C2833', 2:'1C2833', 3:'1C2833', 4:'1C2833', 5:'1C2833', 6:'1A6B6B',
    # Urgency block
    7:'C0392B', 8:'C0392B', 9:'C0392B', 10:'C0392B', 11:'C0392B', 12:'C0392B',
    # Misdirection block
    13:'5B4BA0', 14:'5B4BA0', 15:'5B4BA0', 16:'5B4BA0', 17:'5B4BA0', 18:'5B4BA0',
    # Obstruction block
    19:'D4860B', 20:'D4860B', 21:'D4860B', 22:'D4860B', 23:'D4860B', 24:'D4860B',
    # Summary
    25:'1A5276', 26:'1A5276', 27:'1A5276',
}

ws.row_dimensions[3].height = 36
for col_idx, bg in col_colors.items():
    from openpyxl.utils import get_column_letter
    cell = ws.cell(row=3, column=col_idx)
    hdr(cell, bg=bg)

nrows = len(results_df)
for r in range(4, 4 + nrows):
    ws.row_dimensions[r].height = 38
    row_bg = 'FFFFFF' if r % 2 == 0 else 'F2F3F4'

    for c in range(1, 28):
        cell = ws.cell(row=r, column=c)
        val = str(cell.value or '')

        # Yes/No detection columns (7,13,19) = detected columns
        if c in (7, 13, 19):
            if val == 'Yes':
                dat(cell, bg='D5F5E3', fg='1E8449', bold=True, center=True)
            else:
                dat(cell, bg=row_bg, fg='7F8C8D', center=True)
        # Confidence columns (8,14,20)
        elif c in (8, 14, 20):
            try:
                v = float(val)
                if v >= 0.7:   dat(cell, bg='D5F5E3', fg='1E8449', bold=True, center=True)
                elif v >= 0.4: dat(cell, bg='FDEBD0', fg='D4860B', bold=False, center=True)
                else:          dat(cell, bg=row_bg, fg='7F8C8D', center=True)
            except: dat(cell, bg=row_bg, center=True)
        # Agreement columns (12,18,24)
        elif c in (12, 18, 24):
            if val == 'Match':    dat(cell, bg='D5F5E3', fg='1E8449', bold=True, center=True)
            elif val == 'Mismatch': dat(cell, bg='FADBD8', fg='C0392B', bold=True, center=True)
            else: dat(cell, bg=row_bg, center=True)
        # Website column (3)
        elif c == 3:
            site_map = {'SHEIN':('FADBD8','C0392B'),'AMAZON':('FDEBD0','D4860B'),
                       'ALIEXPRESS':('FDEBD0','D4860B'),'BOOKING':('EAE7F5','5B4BA0'),
                       'EBAY':('D5F5E3','1E8449'),'TEMU':('FADBD8','C0392B')}
            bg_s, fg_s = site_map.get(val, (row_bg, '2C3E50'))
            dat(cell, bg=bg_s, fg=fg_s, bold=True, center=True)
        # ID (1)
        elif c == 1:
            dat(cell, bg='ECF0F1', fg='1A5276', bold=True, center=True)
        # Dominant pattern (25)
        elif c == 25:
            pmap = {'fake_urgency':('FADBD8','C0392B'),
                    'visual_misdirection':('EAE7F5','5B4BA0'),
                    'obstruction':('FDEBD0','D4860B'),
                    'none':('F2F3F4','7F8C8D')}
            bg_p, fg_p = pmap.get(val, ('F2F3F4','7F8C8D'))
            dat(cell, bg=bg_p, fg=fg_p, bold=True, center=True)
        else:
            dat(cell, bg=row_bg)

# Column widths sheet 1
widths = [5,28,12,22,10,30, 8,8,22,22,35,12, 8,8,22,22,35,12, 8,8,22,22,35,12, 20,8,8]
for i, w in enumerate(widths, 1):
    from openpyxl.utils import get_column_letter
    ws.column_dimensions[get_column_letter(i)].width = w

ws.freeze_panes = 'G4'

# ── Sheet 2: Evaluation Metrics ───────────────────────────────────────────────
ws2 = wb['Evaluation Metrics']
ws2.merge_cells('A1:I1')
t2 = ws2['A1']
t2.value = 'Rule-Based Detection Evaluation Metrics — vs Manual Labels'
t2.font = Font(name='Arial', bold=True, size=13, color='FFFFFF')
t2.fill = fill('1C2833')
t2.alignment = Alignment(horizontal='center', vertical='center')
ws2.row_dimensions[1].height = 26

ws2.merge_cells('A2:I2')
s2 = ws2['A2']
s2.value = 'Baseline performance of rule-based system — use this table directly in your thesis results chapter'
s2.font = Font(name='Arial', italic=True, size=9, color='7F8C8D')
s2.fill = fill('F2F3F4')
s2.alignment = Alignment(horizontal='center', vertical='center')
ws2.row_dimensions[2].height = 14

hdr_colors2 = ['1A5276','C0392B','5B4BA0','D4860B','1A5276','1A6B6B','1A6B6B','1A6B6B','1A6B6B']
ws2.row_dimensions[3].height = 24
for ci, bg in enumerate(hdr_colors2, 1):
    hdr(ws2.cell(row=3, column=ci), bg=bg)

pat_colors = {
    'Fake Urgency':      ('FADBD8','C0392B'),
    'Vis. Misdirection': ('EAE7F5','5B4BA0'),
    'Obstruction':       ('FDEBD0','D4860B'),
}
for ri in range(4, 4 + len(metrics_df)):
    ws2.row_dimensions[ri].height = 22
    pat = str(ws2.cell(row=ri, column=1).value or '')
    bg_p, fg_p = pat_colors.get(pat, ('F2F3F4','2C3E50'))
    for ci in range(1, 10):
        cell = ws2.cell(row=ri, column=ci)
        if ci == 1:
            dat(cell, bg=bg_p, fg=fg_p, bold=True)
        else:
            try:
                v = float(cell.value or 0)
                if ci >= 6:  # metric columns
                    if v >= 0.8:   dat(cell, bg='D5F5E3', fg='1E8449', bold=True, center=True)
                    elif v >= 0.6: dat(cell, bg='FDEBD0', fg='D4860B', center=True)
                    else:          dat(cell, bg='FADBD8', fg='C0392B', center=True)
                else:
                    dat(cell, bg='F2F3F4', center=True)
            except:
                dat(cell, center=True)

for ci, w in enumerate([22,6,6,6,6,10,10,10,10], 1):
    ws2.column_dimensions[get_column_letter(ci)].width = w

wb.save(output_path)
print(f'✅ Results saved to {output_path}')

✅ Results saved to rule_based_detection_results.xlsx


In [ ]:
from google.colab import files

files.download('rule_based_detection_results.xlsx')
print('📥 Download started — check your Downloads folder')
print()
print('📌 WHAT YOU HAVE NOW:')
print('   Sheet 1: Full detection results — one row per screenshot')
print('            Each row shows: RB detected Yes/No, confidence score,')
print('            keyword evidence, visual evidence, explanation,')
print('            and whether it agrees with your manual label')
print()
print('   Sheet 2: Evaluation metrics — Precision, Recall, F1, Accuracy')
print('            for each dark pattern category')
print('            → Copy this table directly into your thesis Chapter 5')
print()
print('📌 NEXT STEPS FOR YOUR THESIS:')
print('   1. Review mismatches — they reveal where rules need improvement')
print('   2. Add more keywords to URGENCY_KEYWORDS / MISDIRECTION_KEYWORDS')
print('   3. Re-run after adding more annotated images')
print('   4. This rule-based output becomes your BASELINE')
print('      → Compare it against CLIP zero-shot in Month 3')
print('      → Compare it against fine-tuned ViT in Month 4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started — check your Downloads folder

📌 WHAT YOU HAVE NOW:
   Sheet 1: Full detection results — one row per screenshot
            Each row shows: RB detected Yes/No, confidence score,
            keyword evidence, visual evidence, explanation,
            and whether it agrees with your manual label

   Sheet 2: Evaluation metrics — Precision, Recall, F1, Accuracy
            for each dark pattern category
            → Copy this table directly into your thesis Chapter 5

📌 NEXT STEPS FOR YOUR THESIS:
   1. Review mismatches — they reveal where rules need improvement
   2. Add more keywords to URGENCY_KEYWORDS / MISDIRECTION_KEYWORDS
   3. Re-run after adding more annotated images
   4. This rule-based output becomes your BASELINE
      → Compare it against CLIP zero-shot in Month 3
      → Compare it against fine-tuned ViT in Month 4


### False Negatives for Fake Urgency

These are instances where **Fake Urgency was manually labeled 'Yes'**, but **NOT detected by the rule-based system** (i.e., `RB_urgency_detected` is 'No'). Examining these will help us identify gaps in the `URGENCY_KEYWORDS` or `VISUAL_URGENCY_SIGNALS`.

In [ ]:
# Filter for False Negatives for Fake Urgency
fn_urgency = eval_df[(eval_df['manual_urgency'] == 'Yes') & (eval_df['RB_urgency_detected'] == 'No')]

if not fn_urgency.empty:
    print(f"Found {len(fn_urgency)} False Negatives for Fake Urgency:")
    # Display relevant columns for analysis
    display(fn_urgency[['image_id', 'website', 'ocr_text_preview', 'RB_urgency_explanation']])
else:
    print("No False Negatives found for Fake Urgency. All manually labeled 'Fake Urgency' instances were detected (or there are no manual labels for urgency).")

Found 37 False Negatives for Fake Urgency:


,image_id,website,ocr_text_preview,RB_urgency_explanation
2,3,,,No signals detected
3,4,,,No signals detected
5,6,,,No signals detected
6,7,,,No signals detected
7,8,,,No signals detected
8,9,,,No signals detected
9,10,,,No signals detected
10,11,,,No signals detected
11,12,,,No signals detected
12,13,,,No signals detected
